# CSRNet Training for Crowd Counting (ShanghaiTech)

Notebook này kết hợp các file `model.py`, `dataset.py` và `train.py` để bạn có thể chạy trực tiếp trên Google Colab.

In [ ]:
# 1. Import các thư viện cần thiết
import os
import glob
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from tqdm import tqdm

## 2. Định nghĩa Model (CSRNet)

In [ ]:
class CSRNet(nn.Module):
    def __init__(self, load_weights=False):
        super(CSRNet, self).__init__()
        self.seen = 0
        self.frontend_feat = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512]
        self.backend_feat  = [512, 512, 512, 256, 128, 64]
        self.frontend = self.make_layers(self.frontend_feat)
        self.backend = self.make_layers(self.backend_feat, in_channels=512, dilation=True)
        self.output_layer = nn.Conv2d(64, 1, kernel_size=1)
        
        if load_weights:
            mod = models.vgg16(pretrained=True)
            self._initialize_weights()
            for i in range(len(self.frontend.state_dict().items())):
                list(self.frontend.state_dict().items())[i][1].data[:] = list(mod.state_dict().items())[i][1].data[:]
        else:
            self._initialize_weights()

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        x = self.output_layer(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def make_layers(self, cfg, in_channels=3, batch_norm=False, dilation=False):
        d_rate = 2 if dilation else 1
        layers = []
        for v in cfg:
            if v == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=d_rate, dilation=d_rate)
                if batch_norm:
                    layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                else:
                    layers += [conv2d, nn.ReLU(inplace=True)]
                in_channels = v
        return nn.Sequential(*layers)

## 3. Định nghĩa Dataset

In [ ]:
class CrowdDataset(Dataset):
    def __init__(self, root_dir, part='A', split='train', transform=None):
        self.root_dir = root_dir
        self.part = part
        self.split = split
        self.transform = transform
        self.img_paths = sorted(glob.glob(os.path.join(root_dir, f"part_{part}", f"{split}_data", "images", "*.jpg")))
        
        if self.transform is None:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        h5_path = img_path.replace('.jpg', '.h5').replace('images', 'ground-truth')
        img = Image.open(img_path).convert('RGB')
        with h5py.File(h5_path, 'r') as hf:
            target = np.asarray(hf['density'])
        if self.transform: img = self.transform(img)
        target = torch.from_numpy(target).unsqueeze(0)
        return img, target

## 4. Huấn luyện (Training)

In [ ]:
def train(DATA_ROOT, trial=False):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Sử dụng thiết bị: {device}")
    
    model = CSRNet(load_weights=True).to(device)
    criterion = nn.MSELoss(reduction='sum').to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-5)
    
    train_dataset = CrowdDataset(DATA_ROOT, part='A', split='train')
    if trial: train_dataset.img_paths = train_dataset.img_paths[:5]
    
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    epochs = 1 if trial else 10
    
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for img, target in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            img, target = img.to(device), target.to(device)
            output = model(img)
            target_resized = F.interpolate(target, size=(output.shape[2], output.shape[3]), mode='bilinear') * 64.0
            
            loss = criterion(output, target_resized)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        print(f"Epoch {epoch+1} Avg Loss: {epoch_loss/len(train_loader):.4f}")
        torch.save(model.state_dict(), "csrnet_last.pth")

In [ ]:
# Đường dẫn tới folder ShanghaiTech trên Drive/Colab của bạn
DATA_PATH = "/content/DO-AN-THI-GIAC-MAY-TINH/data/ShanghaiTech"
train(DATA_PATH, trial=True)